<a href="https://colab.research.google.com/github/swarnamalyamohan/enterprise-incident-rag/blob/main/notebooks/incident_rag_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!git clone https://github.com/swarnamalyamohan/enterprise-incident-rag.git
%cd enterprise-incident-rag

Cloning into 'enterprise-incident-rag'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 29 (delta 2), reused 28 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 6.52 KiB | 6.52 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/enterprise-incident-rag/enterprise-incident-rag


In [11]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.2 MB/s eta 0:00:00


In [12]:
import os
import sys
from dotenv import load_dotenv
from google.colab import userdata



# Load environment variables from a .env file
load_dotenv()

# Set the OpenAI API key environment variable (comment out if not using OpenAI)
if not userdata.get('OPENAI_API_KEY'):
    os.environ["OPENAI_API_KEY"] = input("Please enter your OpenAI API key: ")
else:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [14]:
from incident_rag import IncidentRAGPipeline

In [15]:
pipeline = IncidentRAGPipeline()

pipeline.build_knowledge_base("incidents")

Loading incident documents...
Loaded 8 incident chunks
Generating embeddings...
Building local FAISS index...
FAISS index created with 8 vectors
Knowledge base is ready


In [16]:
new_incident = """
High CPU usage detected on payment-service.
Users are reporting slow checkout and payment failures.
CPU crossed 90% for 10 minutes after a recent deployment.
"""

In [17]:
similar_incidents = pipeline.retrieve_similar_incidents(
    new_incident=new_incident,
    top_k=3
)

similar_incidents

[{'incident_id': 'INC-2026-001',
  'service': 'payment-service',
  'severity': 'Critical',
  'date': '2024-03-15',
  'section': 'summary',
  'text': 'Payment service experienced high CPU utilization and increased checkout latency.',
  'source_file': 'INC-2026-001.md',
  'score': 0.7707868218421936},
 {'incident_id': 'INC-2026-001',
  'service': 'payment-service',
  'severity': 'Critical',
  'date': '2024-03-15',
  'section': 'root_cause',
  'text': 'A recent deployment introduced inefficient database polling logic, causing CPU usage to spike above 90%.',
  'source_file': 'INC-2026-001.md',
  'score': 0.7214586138725281},
 {'incident_id': 'INC-2026-001',
  'service': 'payment-service',
  'severity': 'Critical',
  'date': '2024-03-15',
  'section': 'prevention',
  'text': 'Added CPU-based alerts, improved deployment validation, and introduced load testing for database polling changes.',
  'source_file': 'INC-2026-001.md',
  'score': 0.602612316608429}]

In [18]:
import pandas as pd

similar_df = pd.DataFrame([
    {
        "Incident ID": item["incident_id"],
        "Service": item["service"],
        "Severity": item["severity"],
        "Section": item["section"],
        "Similarity %": round(item["score"] * 100, 2),
        "Text Preview": item["text"][:250]
    }
    for item in similar_incidents
])

similar_df

,Incident ID,Service,Severity,Section,Similarity %,Text Preview
0,INC-2026-001,payment-service,Critical,summary,77.08,Payment service experienced high CPU utilizati...
1,INC-2026-001,payment-service,Critical,root_cause,72.15,A recent deployment introduced inefficient dat...
2,INC-2026-001,payment-service,Critical,prevention,60.26,"Added CPU-based alerts, improved deployment va..."


In [19]:
result = pipeline.generate_triage_note(
    new_incident=new_incident,
    top_k=3
)

print(result["triage_note"])

1. Likely Root Cause  
The recent deployment likely introduced inefficient database polling logic or similar code changes that caused CPU usage to spike above 90%, leading to slow checkout and payment failures.

2. Recommended Immediate Actions  
- Roll back the recent deployment to the last known stable version to reduce CPU usage.  
- Investigate the deployment changes focusing on database polling or any CPU-intensive operations.  
- Monitor CPU usage and checkout latency closely after rollback.  
- Notify the development team to prioritize a fix if rollback is not feasible.

3. Similar Past Incidents  
- INC-2026-001 (2024-03-15): High CPU utilization and increased checkout latency in payment-service due to inefficient database polling logic introduced in a recent deployment.

4. Prevention Suggestions  
- Implement CPU-based alerts to detect high utilization early.  
- Improve deployment validation processes, including code reviews focused on database polling and CPU-heavy logic.  

In [20]:
from IPython.display import Markdown, display

display(Markdown(result["triage_note"]))

1. Likely Root Cause  
The recent deployment likely introduced inefficient database polling logic or similar code changes that caused CPU usage to spike above 90%, leading to slow checkout and payment failures.

2. Recommended Immediate Actions  
- Roll back the recent deployment to the last known stable version to reduce CPU usage.  
- Investigate the deployment changes focusing on database polling or any CPU-intensive operations.  
- Monitor CPU usage and checkout latency closely after rollback.  
- Notify the development team to prioritize a fix if rollback is not feasible.

3. Similar Past Incidents  
- INC-2026-001 (2024-03-15): High CPU utilization and increased checkout latency in payment-service due to inefficient database polling logic introduced in a recent deployment.

4. Prevention Suggestions  
- Implement CPU-based alerts to detect high utilization early.  
- Improve deployment validation processes, including code reviews focused on database polling and CPU-heavy logic.  
- Introduce load testing specifically targeting database polling changes before deployment.

5. Confidence and Assumptions  
Confidence is high given the strong similarity (0.72-0.77) to a recent critical incident with the same service and symptoms. Assumes the recent deployment included changes related to database polling or similar CPU-intensive logic as in the past incident.